In [3]:
import pandas as pd
import glob, os
from tqdm.notebook import tqdm
import json
import plotly.express as px
import plotly.graph_objects as go
import geopandas as gpd
import numpy as np

pd.set_option('display.max_columns', None)

In [4]:
CODIGO_COMUNIDAD = 28
ELECCIONES_FILES_PATH = "../../data/raw/resultados_electorales/congreso_por_municipio/"
ELECCIONES_COMPLETAS_PATH = "../../data/staging/elecciones_generales_completas.csv"
GEOJSON_PATH = "../../data/raw/mapa_municipios/Municipios_IGN-shp/madrid_municipios_reduced.geojson"
PRECALCULATED_TABLE = True 

# Datos electorales

In [5]:
def clean_column_names(df):
    df.columns = df.columns.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace(':', '').str.replace('(', '')
    df.columns = df.columns.str.replace('.', '').str.replace(')', '').str.replace('-', '_').str.replace(')', '')
    df.columns = df.columns.str.replace("á", "a").str.replace("é", "e").str.replace("í", "i").str.replace("ó", "o").str.replace("ú", "u")
    return df

def map_party_names(df):
    mappings={       
              'cs': 'ciudadanos', 
              "c's": 'ciudadanos',  
              "psoe_progr":"psoe",}
    df = df.rename(columns=mappings)
    return df

def load_data(data_path):
    df = None
    for file in tqdm(glob.glob(data_path + "*.xlsx")):
        df_single = pd.read_excel(file, header=1, skiprows=4)
        year = os.path.splitext(file)[0].split("/")[-1][3:7]
        month = os.path.splitext(file)[0].split("/")[-1][7:9]
        df_single["year"] = int(year)
        df_single["month"] = int(month)
        df_single["yearmonth"] = year+month
        print("yearmonth: ", year+month)
        df_single = clean_column_names(df_single)

        df_single = map_party_names(df_single)
        
#        print(df_single.columns)
        if df is None:
            df = df_single
        else:
            df = pd.concat([df, df_single],axis=0, ignore_index=True, sort=False)

    cols = [c for c in df.columns if "nnamed" not in c]
    df = df[cols]
    df = df.fillna(0)
    print(df.columns)
    return df

def add_custom_vars(df):
    # crear codigo ine
    print("adding CODIGOINE")
    df["CODIGOINE"] = df["codigo_de_provincia"].astype(int).astype(str).str.zfill(2) + df["codigo_de_municipio"].astype(int).astype(str).str.zfill(3)
    return df

In [6]:
if not PRECALCULATED_TABLE:
    df = load_data(ELECCIONES_FILES_PATH)
    df = add_custom_vars(df)
    df.to_csv(ELECCIONES_COMPLETAS_PATH, sep="|")
else:
    df = pd.read_csv(ELECCIONES_COMPLETAS_PATH, sep="|", dtype={"CODIGOINE":str}, index_col=False)
    df = df.drop(columns=["Unnamed: 0"])

In [7]:
key_columns = ['nombre_de_comunidad', 'codigo_de_provincia', 'nombre_de_provincia',
               'codigo_de_municipio', 'nombre_de_municipio', 'poblacion',
               'numero_de_mesas', 'total_censo_electoral', 'total_votantes',
               'votos_validos', 'votos_a_candidaturas','votos_en_blanco','votos_nulos',
               'year', 'month', 'yearmonth', 'CODIGOINE']
party_columns = [x for x in df.columns if x not in key_columns]

In [8]:
df.shape

(121361, 645)

In [9]:
df[party_columns] = df[party_columns].apply(pd.to_numeric)

## Filtrar por Madrid

In [10]:
df_comunidad = df[df["codigo_de_provincia"] == CODIGO_COMUNIDAD]  # 28 = Madrid

In [11]:
df_comunidad.shape

(2680, 645)

# Mapa Comunidad por munipios

In [12]:
#map_path = "../../data/raw/mapa_municipios/Municipios_IGN-shp/Municipios_IGN.shp"
#geodf = gpd.read_file(map_path)
#geodf["COMUNIDAD"] = geodf["CODIGOINE"].apply(lambda x: str(x)[:2])
#geodf = geodf[geodf["COMUNIDAD"] == str(CODIGO_COMUNIDAD).zfill(2)]
#geodf_reduced = geodf[["CODIGOINE","geometry"]]
#geojson_save_path = "../../data/raw/mapa_municipios/Municipios_IGN-shp/madrid_municipios.geojson"
#guardar
#geodf_reduced.to_file(geojson_save_path, driver = "GeoJSON")

In [13]:
#cargar
with open(GEOJSON_PATH) as geofile:
    geojson = json.load(geofile)

# Mapa de ganadores

In [14]:
# creando una columna con el partido con más votos
df_comunidad["ganador"] = df_comunidad[party_columns].idxmax(axis=1)
df_comunidad["ganador"] = df_comunidad.apply(lambda x: None if x[party_columns].sum() == 0 else x["ganador"], axis=1)
df_comunidad = df_comunidad[df_comunidad["ganador"].notnull()]

<venv>/lib/python3.7/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
<venv>/lib/python3.7/site-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  This is separate from the ipykernel package so we can avoid doing imports until


In [15]:
df_comunidad["ganador"].unique()

array(['pp', 'psoe', 'ciudadanos', 'podemos_iu_equo', 'vox', 'ap_pdp',
       'ucd', 'pcoe', 'pst', 'podemos', 'cds', 'iu', 'pce', 'uce', 'ap',
       'fe', 'ap_pdp_pl', 'podemos_iu'], dtype=object)

# Resultados electorales

In [16]:
color_map = {'ucd':"#FFA07A",
             'psoe':"#E52B50", 
             'pp':"#1E90FF", 
             'ap':"#FFFF00",              
             'pce':"#FF0000",              
             'uce':"#FF0000",              
             'ap_pdp':"#FFFF00",              
             'pcoe':"#FF0000",              
             'pst':"#FF0000",              
             'ap_pdp_pl':"#FFFF00",              
             'cds':"#66CDAA",              
             'iu':"#32CD32",
             'fe':"#000000", 
             'ciudadanos':"#FFA500",
             'podemos_iu':"#9370DB",    
             'podemos_iu_equo':"#9370DB",                 
             'podemos':"#9370DB",
             'vox':"#00FF00"}

In [17]:
# filtros
#df_com_elecciones = df_comunidad[df_comunidad["yearmonth"] == ELECCIONES]
df_com_elecciones = df_comunidad.copy()
df_com_elecciones = df_com_elecciones[["ganador", "nombre_de_municipio", "CODIGOINE", "year", "month"]].sort_values(["year", "month"], ascending=True)

In [18]:
df_com_elecciones.shape

(2674, 5)

In [19]:
df_com_elecciones.tail()

,ganador,nombre_de_municipio,CODIGOINE,year,month
103836,psoe,Villavieja del Lozoya ...,28182,2019,11
103837,podemos_iu,Zarzalejo ...,28183,2019,11
103838,psoe,Lozoyuela-Navas-Sieteiglesias ...,28901,2019,11
103839,pp,Puentes Viejas ...,28902,2019,11
103840,pp,Tres Cantos ...,28903,2019,11


# Gráficos

In [ ]:
## GRAFICO --------------------
#Forzar ganadores en el primer frame para que funcione
df_forze_winner = pd.DataFrame({"ganador":df_com_elecciones.ganador.unique(), 
              "nombre_de_municipio":["UNKNOWN"] * len(df_com_elecciones.ganador.unique()),
              "CODIGOINE":["UNKNOWN"] * len(df_com_elecciones.ganador.unique()),
              "year":[1977] * len(df_com_elecciones.ganador.unique()),
              "month":[6] * len(df_com_elecciones.ganador.unique())}) 
df_tmp = df_com_elecciones.append(df_forze_winner, ignore_index = True)
fig = px.choropleth_mapbox(data_frame=df_tmp, geojson=geojson, locations='CODIGOINE', 
                           featureidkey="properties.CODIGOINE", color='ganador', mapbox_style="carto-positron", 
                           hover_data=["nombre_de_municipio", "ganador", "year", "month"],  
                           color_discrete_map=color_map, zoom=7,
                           center={"lat": 40.463667, "lon": -3.74922}, animation_frame="year",
                           animation_group="ganador",height=650,
                           title= f"Ganador por municipio de las Elecciones Generales en la Comunidad de Madrid")
fig.show()

In [ ]:
## GRAFICO --------------------
# una única elección
ELECCIONES = 201911
df_tmp = df_com_elecciones.copy()
df_tmp["yearmonth"] = df_tmp["year"].astype(str).str.zfill(4) + df_tmp["month"].astype(str).str.zfill(2)
df_tmp = df_tmp[df_tmp["yearmonth"] == str(ELECCIONES)]
fig = px.choropleth_mapbox(data_frame=df_tmp, 
                           geojson=geojson, locations='CODIGOINE', 
                           featureidkey="properties.CODIGOINE", color='ganador', mapbox_style="carto-positron", 
                           hover_data=["nombre_de_municipio", "ganador"], zoom=7, color_discrete_map=color_map,
                           center={"lat": 40.463667, "lon": -3.74922}, 
                           title= f"Elecciones Generales {str(ELECCIONES)[:4]}-{str(ELECCIONES)[-2:]} en Madrid")
fig.show()

# Evolución votos

In [20]:
# evolucion partidos mas votados, lineas dejar rastro, filtro por municipio
CODIGOINE_MUNICIPIO="28079"

In [21]:
df_municipio = df_comunidad[df_comunidad.CODIGOINE == CODIGOINE_MUNICIPIO]
nombre_municipio = df_municipio.nombre_de_municipio.unique()[0]
df_municipio.head()

,nombre_de_comunidad,codigo_de_provincia,nombre_de_provincia,codigo_de_municipio,nombre_de_municipio,poblacion,numero_de_mesas,total_censo_electoral,total_votantes,votos_validos,votos_a_candidaturas,votos_en_blanco,votos_nulos,pp,psoe,iu,ciu,eaj_pnv,cc,bng,hb,erc,pa,ea,uv,lve,cha,uc,upv_bn,psm_ene,cdn,lv_gv,prt,pcpe,ph,pas,upl,fea,icv_gorordo,lv,cex,um,tc_pnc,priojano,pec,urcl,na,aun,cr,sos,prepal,frepic,pspc,prcm,fpg,siex,prv,prim,fei,nr,picc,ar,env,pae,pb,pcn,upra,uad,adec,vdpa,n,alas,aba,prgu,lae,dsa,lg,piz,pnm,por,afit,ccv,year,month,yearmonth,ciudadanos,podemos_iu_equo,vox,erc_sobiranistes,ecp_guanyem_el_canvi,jxcat_junts,eh_bildu,podemos_eu_mareas_en_comun_equo,cca_pnc,na+,compromis_2019,prc,pacma,front_republica,recortes_cero_gv,nca,pact,ara_mes_esquerra,gbai,pum+j,en_marea,pcte,el_pi,axsi,pcoe,avant_adelante_los_verdes,eb,cpm,somos_region,pcpa,uig_som_cuides,erpv,izqp,pcpc,ahora_canarias,cxg,ppso,cnv,cex_crex_prex,pr+,p_lib,cilu_linares,andecha_astur,"""jf""",pyln,fia,fe_de_las_jons,f8,solidaria,dpl,union_regionalista,centrados,vou,dp,pdsje_udec,izar,risa,c_21,+mas+,udt,ap_pdp,ucd,pce,cds,fn,pst,ee,psa_pa,pcc,b_psg,upc,ne,se,uce,pce_m_l,eu,eg,ac_cc,upv,cuc,ps,mfe,eupv,aeba,psm,lc_cosi,psa,env_urv,aegi,ce_fcc,fe_jons,fic,pnc,padl,cc1,idg,fe_i,lcr_mc,lcr,aci,pre,pnc1,pdb,mcate,asdc,ir,fa,em,pcu,pc,emk,bia,cir,podemos,en_comu,iu_upec,podemos_com,erc_catsi,dl,podemos_en,upyd,nos,uniocat,recortes_ce,mes,ccd,fe_de_las_j,x_la_izquie,somval,sain,centro_mode,pt,canarias_de,arapv,eu_ex,dn,ifem,en_positiu,cilus,cra,ln,avant,mlgxsi,pfyv,aju,independien,fdee,soluciona,"js,pc",pld,oe,ruiz_mateos,lv_lv,pte_uc,par,aic,cg,psg_eg,av_mec,verde,acn,csd,png_pg,exu,pore,igc,ped,pam,grm,una,ua_cha,ub,ceu,arde,sev,mv,pancal,coa,pee_lv,r_x_c,tagoror,prb,pnem,ppr,lki_emk,pce_ml,cd,un,pnv,pte,psoe_h,ort,erfn,mc_oic,bnpg,c_upc,pg_pog_psg,bean,pcarl,pct,ufv,fjonsa,upn,oce_br,psar___psda,plib,urv,pnpv,unai,pdpcan,smim,psin,ule,e_catala,cic,p_cant,dscc,pprov,lc,ocebr_uce,cna,pippa,poc,cime,fe_uf,uf_fi_at,verdes,ea_eue,le,arm_ade,pap,upca,ua,gil,pgc,pln,e_u,ummp,pec_verde,nps,p_riojano,cng,ag,prex,sees,arcoiris,app,pvpa,mce,atf,aigranc,fpe,aa,fe_a,unida,udla,os,conceyu,piap,abe,pitrcg,if,p_tagoror,mces,us,ince,freixes,ccl,ap,psp_us,pdpc,fdc_edc,udc_idcc,ec_fed,fdi,asdci,aet,an18,rse,fut,caic,esb,pspv,indep,dcv,psg,fjons,afn,indep1,ura,lliga,uan,anepa_cp,fl,div,cja,cups,ccia,uab,uras,fni,ppcan,ms,asa,mfa,anv,indep2,indep3,indep4,dcar,udib,indep5,cup,pscan,faa,pim,indep6,prsv,icu,psde,indep7,indep8,pdg,indep9,indep10,indep11,aeicyu,cuir,cipye,pli,adc,bai,indep12,cai,ic_v,bloc_verds,psm_en,uc_cds,lv_cm,fe,posi,ev_av,pil,es2000,pade,um_inme,ia,blm,fei_fe_2000,pcl,autonomo,ec,pkd,el_papi,a,prex_crex,dg,upz,cnc,peda,li_lit_ci,paviel,ir_pre,usi,pnca,inar,up_can,ural,ccs,csd_l,pf,ppcr,upb,ii,unib,prib,tpc,cives,mash,pdep,an,eh,iu_lv,amaiur,esquerra,equo,cc_nc_pnc,compromis_q,fac,pxc,anticapitalistas,piratacat,espana_2000,rps,hartosorg,pirata,anc,cdl,pcas,uxv,prao,caballas,cxxi,unidad_del_pueblo,andecha,acima,ucit,dec,cyd,prde,internet,pre_r,basta_ya,pdyc,hd,sxt,dne,cc_pnc,ca,na_bai,nc_ccn,aralar,bloc_idpv_ev_ee,unitat_per_les_illes,psd,cenb,ev_ae,pdeal,rcn_nok,ev_lv,aes,e_2000,rc,cva,ei,tc,amd,pdlpea,partido_riojano,abla,extremadura_unida,pxcat,pnf_no_fumadores,uleg,onv,frente,cdes,plci,pleve,li_litci,plrv,centristas_pctr,pcua,ucpic,irv,mupc,puede,piib,pb_ub,ctc,ud_ca,imc,paie,td,cdas,ave,ucl,n_som,axb,ixc,icbur,agruci,phache,ap_pdp_pl,muc,prd,ac_inc,av,upr,pspa,cun,psdc,adei,cl,loc,cn,pmcn,prm,prov,ae_aic,paec,lki,podemos_iu,mas_pais_equo,cup_pr,podemos_eu,cca_pnc_nc,mes_compromis,teruel_existe!,mas_pais,m_pais_cha_equo,mes_esquerra,xav,contigo,pdsje,mas,unidos_si_acps_def,mdyc,aunacv,canda,ecp,podemos_compromis_eupv,cdc,podemos_en_marea_anova_eu,recortes_cero_grupo_verde,bng_nos,si,centro_moderado,ccd_ci,and,jxc,repo,independientes_fia,entaban,alcd,fme,hrts_ln,bloc_ev,"psm_en,eu,ev,er",aralar_zutik,lv_e,lvcm,pcannabis,lvrm,msr,ides,apca,gve,ci,adn,ptpre,up

In [22]:
# los votos a cero desconciertan el gráfico, las elecciones dobles del 2019 se ven juntas, podemos no aparece en las graficas

In [39]:
## GRAFICO --------------------
df_party_series = df_municipio.melt(id_vars=['year', "month"], value_vars=party_columns)
df_party_series = df_party_series.rename(columns={"variable":"partido", "value":"votos"})
df_party_series = df_party_series[~((df_party_series.year == 2019) & (df_party_series.month == 4))]

top_parties = list(df_party_series.groupby("partido")
                     .sum("votos").sort_values("votos", ascending=False)[:10]
                     .reset_index()["partido"].unique())
df_party_series = df_party_series[df_party_series["partido"].isin(top_parties)]
df_party_series = df_party_series.sort_values(["year", "month", "partido"], ascending=True)
df_party_series["size"] = 5
df_party_series[df_party_series.year == 2019]

,year,month,partido,votos,size
1947,2019,11,ap_pdp,0.0,5
8142,2019,11,ap_pdp_pl,0.0,5
1992,2019,11,cds,0.0,5
1047,2019,11,ciudadanos,155080.0,5
42,2019,11,iu,0.0,5
1977,2019,11,pce,0.0,5
2742,2019,11,podemos,0.0,5
12,2019,11,pp,479448.0,5
27,2019,11,psoe,463364.0,5
1962,2019,11,ucd,0.0,5


In [34]:
max_y = int(df_party_series["votos"].max()*1.1)
fig = px.scatter(df_party_series, x="year", y="votos", animation_frame="year",animation_group="partido",
                 size="size",
                 color="partido", range_x=[1976,2020], range_y=[0,max_y], color_discrete_map=color_map,
                 title= f"Elecciones Generales a lo largo del tiempo en {nombre_municipio}")
fig.show()

In [30]:
fig = px.line(df_party_series, x="year", y="votos", color="partido", line_group="partido", hover_name="partido",
        line_shape="linear", render_mode="svg", color_discrete_map=color_map,range_x=[1976,2020],
        title= f"Elecciones Generales a lo largo del tiempo en {nombre_municipio}")
fig.show()